### Day1下午：MNIST手写数字分类

In [ ]:
for epoch in range(5):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.cuda(), labels.cuda()

        output = model(imgs)
        loss = criterion(output, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.cuda(), labels.cuda()
            output = model(imgs)
            pred = output.argmax(dim=1)
            correct += (pred == labels).sum().item()

    acc = correct / len(val_data)
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/5  Loss: {avg_loss:.4f}  Val Acc: {acc:.4f}")
```

---

## `for epoch in range(5)`

epoch 就是"把整个训练集完整过一遍"。

range(5) 就是过 5 遍。每过一遍，模型就更新一次，变得更准一点。

类比：你做了一套题，对完答案改错，这是 1 个 epoch。做 5 遍就是 5 个 epoch。

---

## `model.train()`

告诉模型"现在是训练模式"。

这个调用会影响两个东西：`Dropout`（训练时随机丢掉一些神经元）和 `BatchNorm`（训练和验证时行为不同）。今天的模型没用这两个，所以暂时不影响结果，但这是必须养成的好习惯。

---

## `for imgs, labels in train_loader`

DataLoader 每次从数据集里取出一个 batch，这里 batch_size=64，所以：

- `imgs` 的形状是 `[64, 1, 28, 28]`（64张图，1个通道，28×28像素）
- `labels` 的形状是 `[64]`（64个数字，0-9）

整个训练集 60000 张，每次取 64 张，所以这个循环会跑 60000÷64 ≈ 937 次。

---

## `imgs, labels = imgs.cuda(), labels.cuda()`

把数据从内存搬到 GPU 显存。

模型在 GPU 上（因为你写了 `.cuda()`），数据也必须在 GPU 上，两者必须在同一个设备，否则报错。

---

## `output = model(imgs)`

前向传播，数据流过模型的每一层，得到预测结果。

output 的形状是 `[64, 10]`，64个样本，每个样本对应10个类别的得分。得分最高的那个类别就是模型的预测。

---

## `loss = criterion(output, labels)`

计算预测值和真实值之间的差距，得到一个数字。

这里用的是 `CrossEntropyLoss`，它做两件事：先把 output 的10个得分转成概率（softmax），再计算正确类别的概率有多高，概率越高 loss 越低。

比如某张图是数字 3，模型给出的10个得分里，如果第3个得分最高，loss 就小；如果第7个得分最高，loss 就大。

---

## `optimizer.zero_grad()`

清空上一步计算的梯度。

PyTorch 默认会把梯度累加，不清空的话本次算出的梯度会加在上次的梯度上，参数更新就乱了。所以每次计算新梯度之前必须清零。

顺序必须是：**先清零，再 backward，再 step**，不能乱。

---

## `loss.backward()`

反向传播，从 loss 出发，用链式法则算出模型里每一个参数对 loss 的梯度，存到每个参数的 `.grad` 属性里。

这一步之后，模型里每个参数都知道了"自己增大一点，loss 会变多少"。

---

## `optimizer.step()`

用刚才算好的梯度更新所有参数。

Adam 优化器在这里做的事：
```
参数 = 参数 - 学习率 × 梯度
```

这一步之后，所有参数都往让 loss 减小的方向移动了一小步。

---

## `total_loss += loss.item()`

把这个 batch 的 loss 累加起来，最后算平均。

`.item()` 是把 GPU 上的 Tensor 变成普通 Python 数字，不加的话没法做加法。

---

## `model.eval()`

切换到验证模式，关掉 Dropout，让 BatchNorm 用统计值而不是当前 batch 的值。验证时必须切回来。

---

## `with torch.no_grad()`

关闭自动求导。

验证时不需要计算梯度，不需要更新参数，关掉自动求导能节省约一半的显存，速度也更快。

---

## `pred = output.argmax(dim=1)`

取每个样本得分最高的类别作为预测结果。

output 是 `[64, 10]`，argmax(dim=1) 沿着类别维度取最大值的下标，结果是 `[64]`，每个数字是 0-9 之间的预测类别。

---

## `correct += (pred == labels).sum().item()`

统计这个 batch 里预测正确的数量。

`pred == labels` 得到 `[64]` 的 True/False，`.sum()` 把 True 加起来（True=1，False=0），`.item()` 转成 Python 数字，累加到 correct 里。

---

## `acc = correct / len(val_data)`

correct 是整个验证集预测正确的总数，除以验证集大小 10000，得到准确率。

---

## `avg_loss = total_loss / len(train_loader)`

total_loss 是所有 batch 的 loss 之和，除以 batch 数量（937），得到平均 loss。

---

整个训练循环的结构就是：
```
外层循环（epoch）：把数据集过几遍
  训练阶段：
    内层循环（batch）：每次取一小批数据
      前向传播 → 算loss → 清零梯度 → 反向传播 → 更新参数
  验证阶段：
    跑一遍验证集，不更新参数，只看准确率